# Lab 6 - Object Detection

Dataset link: https://www.kaggle.com/datasets/biancaferreira/african-wildlife

### Import Libraries

In [14]:
from pathlib import Path
from collections import Counter
import random
import shutil
import yaml
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

### Prepare data

1. Load Dataset

In [15]:
dataset_path = Path(r"D:\COS30082\labs\lab06\elephant")

image_extensions = [".jpg", ".jpeg", ".png"]
image_files = []

for ext in image_extensions:
    image_files.extend(dataset_path.glob(f"*{ext}"))

pairs = [
    (img, img.with_suffix(".txt"))
    for img in image_files
    if img.with_suffix(".txt").exists()
]

print("Images found:", len(image_files))
print("Valid image-label pairs:", len(pairs))

class_counts = Counter()

for img, label in pairs:
    for line in label.read_text().splitlines():
        values = line.split()
        if len(values) == 5:
            class_counts[int(values[0])] += 1

print("Object counts by class ID:", class_counts)

Images found: 376
Valid image-label pairs: 376
Object counts by class ID: Counter({1: 740, 3: 12})


2. Preprocess Data

In [34]:
random.seed(42)
random.shuffle(pairs)

total = len(pairs)
train_end = int(total * 0.80)
val_end = int(total * 0.90)

splits = {
    "train": pairs[:train_end],
    "val": pairs[train_end:val_end],
    "test": pairs[val_end:]
}

output_path = Path(r"D:\COS30082\labs\lab06\elephant_yolo")

if output_path.exists():
    shutil.rmtree(output_path)

# Map original class IDs to new consecutive IDs
class_mapping = {
    1: 0,  # Elephant
    3: 1   # Zebra
}

for split_name, split_pairs in splits.items():
    image_dir = output_path / "images" / split_name
    label_dir = output_path / "labels" / split_name

    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    for img, label in split_pairs:
        shutil.copy2(img, image_dir / img.name)

        new_lines = []

        for line in label.read_text().splitlines():
            values = line.split()

            if len(values) != 5:
                continue

            class_id = int(values[0])

            if class_id not in class_mapping:
                raise ValueError(
                    f"Unexpected class ID {class_id} in {label.name}"
                )

            # Remap the original class ID
            values[0] = str(class_mapping[class_id])
            new_lines.append(" ".join(values))

        (label_dir / label.name).write_text(
            "\n".join(new_lines) + ("\n" if new_lines else ""),
            encoding="utf-8"
        )

# Create data.yaml
data_config = {
    "path": str(output_path.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: "elephant",
        1: "zebra"
    }
}

yaml_path = output_path / "data.yaml"

with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

for split_name, split_pairs in splits.items():
    print(f"{split_name}: {len(split_pairs)} images")

print("Dataset preprocessing complete.")
print("Classes: 0 = elephant, 1 = zebra")
print(f"Dataset configuration: {yaml_path}")

train: 300 images
val: 38 images
test: 38 images
Dataset preprocessing complete.
Classes: 0 = elephant, 1 = zebra
Dataset configuration: D:\COS30082\labs\lab06\elephant_yolo\data.yaml


### Train model

In [35]:
model = YOLO("yolo11n.pt")

results = model.train(
    data=str(yaml_path),
    epochs=30,
    imgsz=640,
    batch=8,
    patience=20,
    seed=42,
    workers=0,
    project="runs/animal_detection",
    name="yolo11n",
    plots=True,
    save=True
)

Ultralytics 8.4.183  Python-3.14.3 torch-2.14.1+cpu CPU (AMD Ryzen 7 8845HS w/ Radeon 780M Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\COS30082\labs\lab06\elephant_yolo\data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n, nbs=64, nms=None, 

### Testing fine-tuned YOLO

In [ ]:
best_model = YOLO(r"D:\COS30082\labs\lab06\runs\detect\runs\animal_detection\yolo11n\weights\best.pt")

test_results = best_model.val(
    data=str(yaml_path),
    split="test",
    imgsz=640,
    batch=8,
    workers=0
)

# Print overall metrics
print("FINE-TUNED YOLO11n — ELEPHANT & ZEBRA")
print(f"Precision:  {test_results.box.mp:.4f}")
print(f"Recall:     {test_results.box.mr:.4f}")
print(f"mAP50:      {test_results.box.map50:.4f}")
print(f"mAP50-95:   {test_results.box.map:.4f}")

# Print per-class AP
print("\nPer-class AP:")
for class_id, ap50, ap in zip(
    test_results.box.ap_class_index,
    test_results.box.ap50,
    test_results.box.ap
):
    class_name = test_results.names[int(class_id)]
    print(f"{class_name}: AP50 = {ap50:.4f}, AP50-95 = {ap:.4f}")

Ultralytics 8.4.183  Python-3.14.3 torch-2.14.1+cpu CPU (AMD Ryzen 7 8845HS w/ Radeon 780M Graphics)
YOLO11n summary (fused): 101 layers, 2,597,294 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.0±0.0 ms, read: 1242.3±213.0 MB/s, size: 129.3 KB)
val: Scanning D:\COS30082\labs\lab06\elephant_yolo\labels\test.cache... 38 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 38/38 11.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 3.6it/s 1.4s0.4s
                   all         38         91      0.733      0.928      0.897      0.732
              elephant         38         90      0.519      0.856      0.798      0.569
                 zebra          1          1      0.946          1      0.995      0.895
Speed: 0.9ms preprocess, 23.9ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to D:\COS30082\labs\lab06\runs\detect\val-3
FINE-TUNED YOLO11n — ELEPHANT & ZEBRA
Precision

### Testing base YOLO

In [39]:
pretrained_model = YOLO(r"D:\COS30082\labs\lab06\yolo11n.pt")

# Original test images and labels
test_image_dir = output_path / "images" / "test"
test_label_dir = output_path / "labels" / "test"

# Create a separate evaluation dataset
eval_path = output_path / "pretrained_eval"

if eval_path.exists():
    shutil.rmtree(eval_path)

eval_images = eval_path / "images" / "test"
eval_labels = eval_path / "labels" / "test"

eval_images.mkdir(parents=True, exist_ok=True)
eval_labels.mkdir(parents=True, exist_ok=True)

# Map custom class IDs to COCO class IDs
class_mapping = {
    0: 20,  # Elephant
    1: 22   # Zebra
}

# Copy test images and remap ground-truth labels
for image_path in test_image_dir.iterdir():
    if image_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
        shutil.copy2(image_path, eval_images / image_path.name)

        label_path = test_label_dir / (image_path.stem + ".txt")
        new_label_path = eval_labels / label_path.name

        new_lines = []

        if label_path.exists():
            for line in label_path.read_text().splitlines():
                values = line.split()

                if len(values) != 5:
                    continue

                original_class_id = int(values[0])

                if original_class_id not in class_mapping:
                    raise ValueError(
                        f"Unexpected class ID {original_class_id} "
                        f"in {label_path.name}"
                    )

                values[0] = str(class_mapping[original_class_id])
                new_lines.append(" ".join(values))

        new_label_path.write_text(
            "\n".join(new_lines) + ("\n" if new_lines else ""),
            encoding="utf-8"
        )

# Preserve COCO's original 80 class indices
coco_names = [f"class_{i}" for i in range(80)]
coco_names[20] = "elephant"
coco_names[22] = "zebra"

eval_yaml = eval_path / "data.yaml"

with open(eval_yaml, "w", encoding="utf-8") as f:
    yaml.safe_dump({
        "path": str(eval_path.resolve()),
        "train": "images/test",
        "val": "images/test",
        "test": "images/test",
        "names": coco_names
    }, f, sort_keys=False)

# Evaluate both elephant and zebra classes
pretrained_results = pretrained_model.val(
    data=str(eval_yaml),
    split="test",
    classes=[20, 22],
    imgsz=640,
    batch=8,
    workers=0,
    plots=True,
    project="runs/pretrained_evaluation",
    name="yolo11n_elephant_zebra"
)

# Display overall metrics
print("PRETRAINED YOLO11n — ELEPHANT & ZEBRA")
print(f"Precision:  {pretrained_results.box.mp:.4f}")
print(f"Recall:     {pretrained_results.box.mr:.4f}")
print(f"mAP50:      {pretrained_results.box.map50:.4f}")
print(f"mAP50-95:   {pretrained_results.box.map:.4f}")

# Display per-class AP
print("\nPer-class AP:")
for class_id, ap50, ap in zip(
    pretrained_results.box.ap_class_index,
    pretrained_results.box.ap50,
    pretrained_results.box.ap
):
    class_name = pretrained_results.names[int(class_id)]
    print(f"{class_name}: AP50 = {ap50:.4f}, AP50-95 = {ap:.4f}")

Ultralytics 8.4.183  Python-3.14.3 torch-2.14.1+cpu CPU (AMD Ryzen 7 8845HS w/ Radeon 780M Graphics)
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs
val: Fast image access  (ping: 0.0±0.0 ms, read: 93.6±158.8 MB/s, size: 1137.4 KB)
val: Scanning D:\COS30082\labs\lab06\elephant_yolo\pretrained_eval\labels\test... 38 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 38/38 805.1it/s 0.0s
val: New cache created: D:\COS30082\labs\lab06\elephant_yolo\pretrained_eval\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 3.6it/s 1.4s0.4s
                   all         38         91      0.907      0.867      0.969       0.84
              elephant         38         90      0.925      0.733      0.942      0.785
                 zebra          1          1      0.889          1      0.995      0.895
Speed: 0.8ms preprocess, 24.9ms inference, 0.0ms loss, 0.9ms postprocess per image
R